# Hand-engineered features y reconocimiento robusto de objetos

## Autores

* **Juan Diego Gallego Nicolás (jdiego.gallego@um.es)**
* **Óscar Vera López (oscar.veral@um.es)**

# Introducción al notebook

Este notebook es la cuarta parte de una serie de cinco que constituye nuestra práctica para la asignatura de Visión Artificial del Máster en Inteligencia Artificial de la Universidad de Murcia. Alguno de los métodos/resultados que aquí se utilizan pueden estar comentados en un notebook anterior. Para el correcto funcionamiento del mismo es necesario instalar el paquete del proyecto siguiendo las instrucciones del [repositorio de GitHub](https://github.com/oscarveral/vision.git).

TODO. Descripcion del contenido del notebook.

Al final del documento, incluimos un apartado de conclusiones y justificación de los ítems de bloques cubiertos por nuestro trabajo. También dejamos un párrafo explicando el papel de la IA generativa en la elaboración del mismo.

# Configuración e inicialización

## Importación de librerías

En el siguiente fragmento de código se incluyen las importaciones necesarias para la ejecución de todos los bloques de código del notebook. Recuerde crear un entorno virtual con el proyecto instalado (Ejecutar "pip install -e ." en el directorio raíz del proyecto) y seguir las instrucciones del fichero README.md.

In [13]:
!pip install ultralytics
!pip install Pillow numpy
!pip install pandas
!pip install matplotlib
!pip install seaborn
!pip install wandb

  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 1.8 MB/s eta 0:00:002.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 2.6 MB/s eta 0:00:00m eta 0:00:010:00:01m
Using cached click-8.3.1-py3-none-any.whl (108 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.2/208.2 kB 2.0 MB/s eta 0:00:000:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 2.2 MB/s eta 0:00:00m eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 2.9 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 3.0 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 2.2 MB/s eta 0:00:00


# Primer entrenamiento con YOLO

Como primer intento, precargamos YOLO8 y lo entrenamos con nuestros datos de test pasados 15 veces.

In [2]:
import os
from ultralytics import YOLO
from pathlib import Path

# --- CONFIGURACIÓN ---
# Apuntamos a la carpeta que creó tu script anterior ('bt4_final')
DATASET_DIR = Path("./images/bt4") 
PROJECT_NAME = "proyecto_trafico"
RUN_NAME = "yolov8_zod_finetuning"
YOLO_DIR = Path("../yolov8_trafico")
if not YOLO_DIR.exists():
    # Crear
    YOLO_DIR.mkdir(parents=True)

# Parámetros Clave
# imgsz=1024 es vital. Las imágenes originales son 3848x2168.
# Si bajas a 640, perderás las señales pequeñas.
IMG_SIZE = 1024   
BATCH_SIZE = 2    # Batch bajo para no saturar la GPU con imágenes grandes
EPOCHS = 15       # Suficiente para ver convergencia con Transfer Learning

# --- 1. GENERAR DATA.YAML ---
# Este archivo le dice a YOLO dónde están las imágenes
yaml_content = f"""
path: {DATASET_DIR.absolute().as_posix()} 
train: images/train
val: images/val

names:
  0: TrafficSign
"""

yaml_path = YOLO_DIR / "dataset_config.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"✔ Configuración creada en: {os.path.abspath(yaml_path)}")

# --- 2. CARGAR MODELO (BT4g) ---
# Usamos yolov8n (nano) para máxima velocidad. 
# Si tienes buena GPU, puedes probar 'yolov8s.pt' (small) o 'yolov8m.pt' (medium).
print("--- Cargando pesos preentrenados (COCO) ---")
model = YOLO('yolov8n.pt') 

# --- 3. ENTRENAMIENTO (BT4c + BT4d) ---
print(f"--- Iniciando entrenamiento en {DATASET_DIR} ---")

# Ultralytics maneja automáticamente:
# - Transfer Learning (congela capas iniciales o ajusta learning rate)
# - Data Augmentation (Mosaic, mixup, cambios de color HSV)
results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    name=RUN_NAME,
    patience=5,        # Parar si no mejora en 5 épocas
    exist_ok=True,     # Sobrescribir si re-ejecutas
    save=True,         # Guardar checkpoints
    plots=True         # Generar gráficas de métricas (BT4b)
)

# --- 4. VALIDACIÓN FINAL (BT4b) ---
print("--- Generando métricas finales ---")
metrics = model.val()

print("\n" + "="*30)
print("RESULTADOS DEL ENTRENAMIENTO")
print("="*30)
print(f"mAP50-95 (Precisión General): {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print("="*30)
print(f"👉 Los gráficos para tu memoria (F1-Curve, Confusion Matrix) están en:")
print(f"   {os.path.join(PROJECT_NAME, RUN_NAME)}")

✔ Configuración creada en: /home/tfg/Desktop/vision/yolov8_trafico/dataset_config.yaml
--- Cargando pesos preentrenados (COCO) ---
--- Iniciando entrenamiento en images/bt4 ---
New https://pypi.org/project/ultralytics/8.3.237 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.236 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24072MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../yolov8_trafico/dataset_config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj

# Segundo entrenamiento con YOLO

Subimos las épocas a 50 y sincronizamos con wandb

In [1]:
import os
import wandb
from ultralytics import YOLO
from pathlib import Path
from wandb import Settings # Importamos Settings para mayor estabilidad de red

# --- LIMPIEZA DE SESIONES ANTERIORES ---
try:
    wandb.finish()
except:
    pass

# --- CONFIGURACIÓN ---
# 1. Apuntamos al dataset FINAL limpio generado anteriormente
DATASET_DIR = Path("./images/bt4") 

PROJECT_NAME = "dgst"
RUN_NAME = "yolov8_nano_200epochs" # Nombre descriptivo para el experimento final
YOLO_DIR = Path("../yolov8_trafico")

if not YOLO_DIR.exists(): YOLO_DIR.mkdir(parents=True)

# Parámetros Optimizados para tu RTX 4090
IMG_SIZE = 1024   
BATCH_SIZE = 16   # Aprovechamos la VRAM de la 4090
EPOCHS = 200      # Entrenamiento largo con early stopping
PATIENCE = 15     # Paciencia para early stopping

# --- 1. GENERAR DATA.YAML ---
yaml_content = f"""
path: {DATASET_DIR.absolute().as_posix()} 
train: images/train
val: images/val
names:
  0: TrafficSign
"""
with open(YOLO_DIR / "dataset_config.yaml", "w") as f:
    f.write(yaml_content)

# --- 2. INICIALIZAR WANDB MANUALMENTE ---
print("--- Inicializando WandB ---")
run = wandb.init(
    project=PROJECT_NAME,
    name=RUN_NAME,
    config={
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "model": "yolov8n",
        "dataset": str(DATASET_DIR)
    },
    reinit=True,
    # Ajuste de seguridad para evitar errores de conexión/pipe en notebooks
    settings=Settings(start_method="fork")
)

# --- 3. DEFINIR CALLBACK PERSONALIZADO ---
def force_wandb_log(trainer):
    """
    Extrae las métricas del entrenador de YOLO y las empuja a W&B a la fuerza.
    """
    log_dict = {}
    
    # 1. Recuperar Pérdidas de Entrenamiento
    if hasattr(trainer, 'loss_items'):
        log_dict["train/box_loss"] = trainer.loss_items[0].item()
        log_dict["train/cls_loss"] = trainer.loss_items[1].item()
        log_dict["train/dfl_loss"] = trainer.loss_items[2].item()
    
    # 2. Recuperar Métricas de Validación
    if hasattr(trainer, 'metrics'):
        log_dict["metrics/precision"] = trainer.metrics.get('metrics/precision(B)', 0)
        log_dict["metrics/recall"] = trainer.metrics.get('metrics/recall(B)', 0)
        log_dict["metrics/mAP50"] = trainer.metrics.get('metrics/mAP50(B)', 0)
        log_dict["metrics/mAP50-95"] = trainer.metrics.get('metrics/mAP50-95(B)', 0)
        log_dict["val/box_loss"] = trainer.metrics.get('val/box_loss', 0)
        log_dict["val/cls_loss"] = trainer.metrics.get('val/cls_loss', 0)
        log_dict["val/dfl_loss"] = trainer.metrics.get('val/dfl_loss', 0)

    # 3. Guardar Learning Rate
    if hasattr(trainer, 'optimizer'):
        log_dict["lr"] = trainer.optimizer.param_groups[0]['lr']

    # Enviar
    wandb.log(log_dict)

# --- 4. CARGAR Y ENTRENAR ---
print("--- Cargando modelo ---")
model = YOLO('yolov8n.pt') 

# Añadimos callback
model.add_callback("on_fit_epoch_end", force_wandb_log)

print(f"--- Iniciando entrenamiento FINAL ({EPOCHS} épocas) ---")
try:
    results = model.train(
        data=YOLO_DIR / "dataset_config.yaml",
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        project=PROJECT_NAME,
        name=RUN_NAME,
        patience=PATIENCE,       # Parar si no mejora en 10 épocas
        exist_ok=True,
        plots=True 
        # No pasamos argumentos wandb aquí porque ya está iniciado arriba
    )
except Exception as e:
    print(f"Error en entrenamiento: {e}")
finally:
    wandb.finish() # Cerrar sesión al terminar

print(f"\nRevisa tus métricas aquí: {run.get_url()}")

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


--- Inicializando WandB ---


wandb: Currently logged in as: juandiego-gallenico (juandiego-gallenico-universidad-de-murcia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


--- Cargando modelo ---
--- Iniciando entrenamiento FINAL (200 épocas) ---
New https://pypi.org/project/ultralytics/8.3.237 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.236 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24072MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../yolov8_trafico/dataset_config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=y

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


lr,▁▅████▇▇▇▇▇▇▇▇▇▇▇▆▆▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄
metrics/mAP50,▁▁▂▃▃▄▅▆▅▆▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇█▇▇██████████
metrics/mAP50-95,▁▁▂▃▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▆▇▇▇▇▇▇▇▇▇████████
metrics/precision,█▂▁▁▂▂▂▂▂▂▂▃▂▃▃▃▃▃▃▃▃▄▄▃▃▃▄▃▄▃▄▄▃▄▄▄▄▄▄▄
metrics/recall,▁▁▂▂▃▅▄▅▄▄▄▅▅▅▆▆▆▆▆▆▆▅▆▆▆▇▆▇▆▇▇▇█▇▇████▇
train/box_loss,▇█▇▄▄▃▅▅▃▄▄▅▃▂▄▁▃▃▂▃▃▂▂▂▄▃▃▂▂▃▃▃▂▂▂▂▁▂▁▅
train/cls_loss,█▆▃▃▄▃▃▄▃▃▃▃▃▃▃▂▂▃▄▃▂▃▂▁▂▂▂▂▁▃▂▂▁▁▂▁▂▁▁▁
train/dfl_loss,▇█▇▃▆▅▃▃▅▆▃▃▅▂▃▁▄▃▃▁▄▃▃▂▄▂▂▅▂▂▃▁▂▂▁▃▃▂▂▁
val/box_loss,████▇▇█▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▁
val/cls_loss,█▅▅▅▅▅▅▅▅▄▄▅▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▁
+1,...


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.



Revisa tus métricas aquí: https://wandb.ai/juandiego-gallenico-universidad-de-murcia/dgst/runs/k4n3rl5w


# Probando con YOLO medium

In [3]:
import os
import wandb
from ultralytics import YOLO
from pathlib import Path
from wandb import Settings # Importamos Settings para mayor estabilidad de red

# --- LIMPIEZA DE SESIONES ANTERIORES ---
try:
    wandb.finish()
except:
    pass

# --- CONFIGURACIÓN ---
# 1. Apuntamos al dataset FINAL limpio generado anteriormente
DATASET_DIR = Path("./images/bt4") 

PROJECT_NAME = "dgst"
RUN_NAME = "yolov8_medium_200_epochs" # Nombre descriptivo para el experimento final
YOLO_DIR = Path("../yolov8_trafico")

if not YOLO_DIR.exists(): YOLO_DIR.mkdir(parents=True)

# Parámetros Optimizados para tu RTX 4090
IMG_SIZE = 1024   
BATCH_SIZE = 8   # Aprovechamos la VRAM de la 4090
EPOCHS = 200     # Entrenamiento largo con early stopping
PATIENCE = 15    # Paciencia para early stopping

# --- 1. GENERAR DATA.YAML ---
yaml_content = f"""
path: {DATASET_DIR.absolute().as_posix()} 
train: images/train
val: images/val
names:
  0: TrafficSign
"""
with open(YOLO_DIR / "dataset_config.yaml", "w") as f:
    f.write(yaml_content)

# --- 2. INICIALIZAR WANDB MANUALMENTE ---
print("--- Inicializando WandB ---")
run = wandb.init(
    project=PROJECT_NAME,
    name=RUN_NAME,
    config={
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "model": "yolov8m",
        "dataset": str(DATASET_DIR)
    },
    reinit=True,
    # Ajuste de seguridad para evitar errores de conexión/pipe en notebooks
    settings=Settings(start_method="fork")
)

# --- 3. DEFINIR CALLBACK PERSONALIZADO ---
def force_wandb_log(trainer):
    """
    Extrae las métricas del entrenador de YOLO y las empuja a W&B a la fuerza.
    """
    log_dict = {}
    
    # 1. Recuperar Pérdidas de Entrenamiento
    if hasattr(trainer, 'loss_items'):
        log_dict["train/box_loss"] = trainer.loss_items[0].item()
        log_dict["train/cls_loss"] = trainer.loss_items[1].item()
        log_dict["train/dfl_loss"] = trainer.loss_items[2].item()
    
    # 2. Recuperar Métricas de Validación
    if hasattr(trainer, 'metrics'):
        log_dict["metrics/precision"] = trainer.metrics.get('metrics/precision(B)', 0)
        log_dict["metrics/recall"] = trainer.metrics.get('metrics/recall(B)', 0)
        log_dict["metrics/mAP50"] = trainer.metrics.get('metrics/mAP50(B)', 0)
        log_dict["metrics/mAP50-95"] = trainer.metrics.get('metrics/mAP50-95(B)', 0)
        log_dict["val/box_loss"] = trainer.metrics.get('val/box_loss', 0)
        log_dict["val/cls_loss"] = trainer.metrics.get('val/cls_loss', 0)
        log_dict["val/dfl_loss"] = trainer.metrics.get('val/dfl_loss', 0)

    # 3. Guardar Learning Rate
    if hasattr(trainer, 'optimizer'):
        log_dict["lr"] = trainer.optimizer.param_groups[0]['lr']

    # Enviar
    wandb.log(log_dict)

# --- 4. CARGAR Y ENTRENAR ---
print("--- Cargando modelo ---")
model = YOLO('yolov8m.pt') 

# Añadimos callback
model.add_callback("on_fit_epoch_end", force_wandb_log)

print(f"--- Iniciando entrenamiento FINAL ({EPOCHS} épocas) ---")
try:
    results = model.train(
        data=YOLO_DIR / "dataset_config.yaml",
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        project=PROJECT_NAME,
        name=RUN_NAME,
        patience=PATIENCE,       # Parar si no mejora en 10 épocas
        exist_ok=True,
        plots=True 
        # No pasamos argumentos wandb aquí porque ya está iniciado arriba
    )
except Exception as e:
    print(f"Error en entrenamiento: {e}")
finally:
    wandb.finish() # Cerrar sesión al terminar

print(f"\nRevisa tus métricas aquí: {run.get_url()}")

--- Inicializando WandB ---


--- Cargando modelo ---
--- Iniciando entrenamiento FINAL (200 épocas) ---
New https://pypi.org/project/ultralytics/8.3.238 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.236 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24072MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../yolov8_trafico/dataset_config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yo

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


lr,████▇▇▇▇▆▆▆▆▆▆▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
metrics/mAP50,▁▄▄▅▅▆▆▇▇▇▇▇▇▇▇▇▇███████████████████████
metrics/mAP50-95,▁▂▃▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█▇▇███████████████
metrics/precision,▁▄▄▄▃▅▅▆▆▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▆▇▇██▇▇▇
metrics/recall,▁▃▃▃▄▅▄▄▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇█▇█▇███▇█▇██▇███
train/box_loss,▄█▆▆▄▅▅▄▃▃▄▃▃▃▃▂▃▂▃▃▃▂▃▁▂▃▃▃▂▄▁▃▂▁▂▃▂▂▂▂
train/cls_loss,███▇▄▅▆▄▄▆▄▄▄▃▄▄▃▃▄▃▃▂▂▃▂▂▃▂▃▃▂▃▂▂▃▂▂▂▁▁
train/dfl_loss,▅▆▅█▃▄▃▃▃▃▂▅▄▄▅▄▃▃▃▅▄▅▄▃▃▃▂▂▃▂▃▂▂▃▂▃▃▂▁▄
val/box_loss,▆██▆▅▃▄▄▃▃▂▂▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/cls_loss,█▇█▇▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▅▆▅▅▆▆▅▅▅▅▅▅▅▅▅▅▅▁
+1,...



Revisa tus métricas aquí: https://wandb.ai/juandiego-gallenico-universidad-de-murcia/dgst/runs/3ijzn8au


# Probar el mejor modelo

In [25]:
import os
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path

# --- 1. CONFIGURACIÓN DE LA PRUEBA ---
IMAGE_ID = "006283"        # ID de la imagen (sin .jpg)
SPLIT = "val"              # Carpeta: 'train' o 'val'

# Rutas (Ajustadas a tu estructura anterior)
DATASET_DIR = Path("./images/bt4")  # O "bt4_final" si usaste el último script
# Ruta al modelo entrenado (Asegúrate de que este archivo existe)
MODEL_N = "medium"
MODEL_PATH = f"dgst/yolov8_{MODEL_N}_200_epochs/weights/best.pt"

# --- 2. LÓGICA DE VISUALIZACIÓN ---
img_path = DATASET_DIR / "images" / SPLIT / f"{IMAGE_ID}.jpg"

if not img_path.exists():
    print(f"❌ Error: No se encuentra la imagen: {img_path}")
    print(f"   Verifica que el ID '{IMAGE_ID}' existe en la carpeta '{SPLIT}'.")

elif not os.path.exists(MODEL_PATH):
    print(f"❌ Error: No se encuentra el modelo en: {MODEL_PATH}")
    print("   Verifica que el entrenamiento haya terminado y generado 'best.pt'.")

else:
    print(f"🤖 Cargando modelo desde: {MODEL_PATH}...")
    model = YOLO(MODEL_PATH)

    print(f"🔎 Realizando inferencia en: {img_path.name}...")
    # conf=0.25 es el umbral estándar. Bájalo si no detecta nada, súbelo si hay ruido.
    results = model.predict(source=img_path, conf=0.25, imgsz=1024, verbose=False)

    # Obtener la imagen con las cajas dibujadas
    # plot() devuelve un array numpy en formato BGR (Blue-Green-Red)
    # Quitar los titulos de comentario innecesarios

    res_plotted = results[0].plot(labels=False)  

    # Convertir de BGR (OpenCV/YOLO) a RGB (Matplotlib) para que los colores se vean bien
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)

    # Mostrar resultado. Quitar bordes blancos
    plt.figure(figsize=(16, 9)) # Tamaño grande para ver detalles 4K
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.imshow(res_rgb)
    plt.axis('off')
    #plt.title(f"Predicción: {IMAGE_ID} ({SPLIT})", fontsize=14)
    plt.savefig(f"inferencia_{MODEL_N}_{IMAGE_ID}.png")

🤖 Cargando modelo desde: dgst/yolov8_medium_200_epochs/weights/best.pt...
🔎 Realizando inferencia en: 006283.jpg...


In [10]:
from ultralytics import YOLO

MODEL_PATH = 'dgst/yolov8_medium_200_epochs/'

# Cargar tu modelo entrenado (el mejor)
model = YOLO(f'{MODEL_PATH}weights/best.pt') 

# 1. Validar con confianza baja (verás que el Recall sube y Falsos Negativos bajan)
print("--- Validando con conf=0.01 ---")
model.val(data="../yolov8_trafico/dataset_config.yaml", conf=0.01, name='val_conf_low')

# 2. Validar con confianza estándar (esto reproducirá tu matriz actual)
print("--- Validando con conf=0.25 ---")
model.val(data="../yolov8_trafico/dataset_config.yaml", conf=0.25, name='val_conf_high')

--- Validando con conf=0.01 ---
Ultralytics 8.3.236 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24072MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 9233.9±2939.1 MB/s, size: 487.4 KB)
val: Scanning /home/tfg/Desktop/vision/notebooks/images/bt4/labels/val.cache... 386 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 386/386 1.3Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 7.8it/s 3.2s<0.1s
                   all        386       3955      0.721      0.512        0.6      0.403
Speed: 1.0ms preprocess, 4.9ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /home/tfg/Desktop/vision/runs/detect/val_conf_low
--- Validando con conf=0.25 ---
Ultralytics 8.3.236 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24072MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x71a16cdb7e00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

# Exportar a ONNX

In [3]:
from ultralytics import YOLO
import os

# 1. Cargar tu mejor modelo reentrenado
model_path = './dgst/yolov8_medium_200_epochs/weights/best.pt'
model = YOLO(model_path)

# 2. Exportar a ONNX
# dynamic=True permite distintos tamaños de imagen (opcional pero recomendado)
success = model.export(format='onnx', dynamic=True)

print(f"Exportación exitosa: {success}")

# 3. Comprobar tamaños de archivo
size_pt = os.path.getsize(model_path) / (1024 * 1024)
size_onnx = os.path.getsize(model_path.replace('.pt', '.onnx')) / (1024 * 1024)

print(f"Tamaño .pt: {size_pt:.2f} MB")
print(f"Tamaño .onnx: {size_onnx:.2f} MB")

import time
import cv2
from ultralytics import YOLO

# Cargar modelos
model_pt = YOLO('./dgst/yolov8_medium_200_epochs/weights/best.pt')
model_onnx = YOLO('./dgst/yolov8_medium_200_epochs/weights/best.onnx')

# Imagen de prueba (una de tu dataset)
img_path = "./images/bt4/images/val/000224.jpg"
img = cv2.imread(img_path)

# Función para medir tiempo (hacemos varias pasadas para calentar)
def medir_inferencia(model, img, nombre, runs=50):
    # Calentamiento
    for _ in range(10):
        model(img, verbose=False)
    
    start = time.time()
    for _ in range(runs):
        model(img, verbose=False)
    end = time.time()
    
    avg_time = (end - start) / runs * 1000 # a ms
    print(f"Tiempo promedio {nombre}: {avg_time:.2f} ms")

print("--- Comparando Inferencia ---")
medir_inferencia(model_pt, img, "PyTorch (.pt)")
medir_inferencia(model_onnx, img, "ONNX Runtime (.onnx)")

Ultralytics 8.3.236 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (13th Gen Intel Core i7-13700KF)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from 'dgst/yolov8_medium_200_epochs/weights/best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 5, 21504) (49.7 MB)

ONNX: starting export with onnx 1.19.1 opset 22...
ONNX: slimming with onnxslim 0.1.80...
ONNX: export success ✅ 1.6s, saved as 'dgst/yolov8_medium_200_epochs/weights/best.onnx' (99.0 MB)

Export complete (2.3s)
Results saved to /home/tfg/Desktop/vision/notebooks/dgst/yolov8_medium_200_epochs/weights
Predict:         yolo predict task=detect model=dgst/yolov8_medium_200_epochs/weights/best.onnx imgsz=1024  
Validate:        yolo val task=detect model=dgst/yolov8_medium_200_epochs/weights/best.onnx imgsz=1024 data=../yolov8_trafico/dataset_config.yaml  
Visualize:       https://netron.app
Exportación exitosa: dgst/yolov8_medium_200_epochs/weights/best.onnx
Tamañ

# Probar equivalencia funcional

In [6]:
import os
import sys
from ultralytics import YOLO
import numpy as np

# --- 1. CONFIGURACIÓN ROBUSTA ---
# Asegúrate de que estas rutas son correctas
PT_MODEL = 'dgst/yolov8_medium_200_epochs/weights/best.pt'
ONNX_MODEL = 'dgst/yolov8_medium_200_epochs/weights/best.onnx'

# Vamos a buscar una imagen válida automáticamente para evitar bloqueos
base_dir = './images/bt4/images/val'
# Lista de intentos por si la 000224 no existe
candidates = ['000224.jpg', '001237.jpg', '006283.jpg'] 
img_path = None

if os.path.exists(base_dir):
    for cand in candidates:
        p = os.path.join(base_dir, cand)
        if os.path.exists(p):
            img_path = p
            break

if img_path is None:
    print(f"⚠️ No encontré imágenes en {base_dir}. Usando imagen de prueba de internet...")
    img_path = "https://ultralytics.com/images/bus.jpg" # Fallback seguro
else:
    print(f"✅ Usando imagen local: {img_path}")

# --- 2. INFERENCIA ---
print("1. Cargando modelo PyTorch...", flush=True)
# Forzamos CPU para evitar conflictos con ONNX en este test
model_pt = YOLO(PT_MODEL)

print("2. Cargando modelo ONNX...", flush=True)
model_onnx = YOLO(ONNX_MODEL)

print("3. Ejecutando inferencia PyTorch (CPU)...")
res_pt = model_pt(img_path, device='cpu', imgsz=1024, conf=0.25, iou=0.45, verbose=False)[0]

print("4. Ejecutando inferencia ONNX (CPU)...")
res_onnx = model_onnx(img_path, device='cpu', imgsz=1024, task='detect', conf=0.25, iou=0.45, verbose=False)[0]

# --- 3. COMPARACIÓN ---
print("\n--- RESULTADOS ---")
boxes_pt = res_pt.boxes.data.cpu().numpy()
boxes_onnx = res_onnx.boxes.data.cpu().numpy()

# Ordenamos por confianza (columna 4) para alinear las detecciones
if len(boxes_pt) > 0:
    boxes_pt = boxes_pt[boxes_pt[:, 4].argsort()[::-1]]
    
if len(boxes_onnx) > 0:
    boxes_onnx = boxes_onnx[boxes_onnx[:, 4].argsort()[::-1]]

print(f"Detecciones PyTorch: {len(boxes_pt)}")
print(f"Detecciones ONNX:    {len(boxes_onnx)}")

if len(boxes_pt) > 0 and len(boxes_onnx) > 0:
    # Comparamos la mejor caja
    top_pt = boxes_pt[0]
    top_onnx = boxes_onnx[0]
    
    diff = np.abs(top_pt - top_onnx)
    max_diff = np.max(diff)
    
    print(f"\nTop 1 PyTorch: {top_pt}")
    print(f"Top 1 ONNX:    {top_onnx}")
    print(f"\n>> Diferencia máxima: {max_diff:.8f}")
    
    if max_diff < 0.001:
        print("✅ EQUIVALENCIA CONFIRMADA")
    else:
        print("⚠️ Hay diferencias numéricas (revisa si son clases distintas)")
else:
    print("⚠️ Uno de los modelos no detectó nada.")

✅ Usando imagen local: ./images/bt4/images/val/000224.jpg
1. Cargando modelo PyTorch...
2. Cargando modelo ONNX...
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
3. Ejecutando inferencia PyTorch (CPU)...
4. Ejecutando inferencia ONNX (CPU)...
Loading dgst/yolov8_medium_200_epochs/weights/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 CPUExecutionProvider

--- RESULTADOS ---
Detecciones PyTorch: 14
Detecciones ONNX:    14

Top 1 PyTorch: [     814.61      628.96      932.28      752.26     0.92083           0]
Top 1 ONNX:    [     814.61      628.96      932.28      752.26     0.92083           0]

>> Diferencia máxima: 0.00000006
✅ EQUIVALENCIA CONFIRMADA
